# <center> <font size = 24 color = 'steelblue'>**Nova Haven - UrbanPulse Analytics**
**Complaint Classification and Routing Recommendations**

### Model 4: 311 Complaint Classification — NLP

- Multi-class text classification: predict the complaint type from text
- **Top 5 categories + "Other" (6 classes total):** Illegal Parking, HEAT/HOT WATER, Noise - Residential, Snow or Ice, Blocked Driveway, and Other (everything else)
- Use NLP techniques (TF-IDF + classifier, embeddings, or transformers)
- Handle real-world citizen text (informal language, abbreviations, multiple languages)
- **Minimum Benchmark:** Accuracy > 75%, weighted F1 > 0.70
- **Stretch Goal:** Accuracy > 85%, weighted F1 > 0.80
- **Also valued:** Complaint routing recommendations

In [1]:

"""
Model 4: NLP Classification — Training Script
=============================================
Clean training script for UrbanPulse 311 complaint classification.

Design choices for this version:
- Fast and stable training
- No multilingual translation during training
- Clear step-by-step logging
- Saves only the artifacts needed by predict.py
"""
from __future__ import annotations

import logging
import random
import re
import sys
import warnings
from pathlib import Path
from typing import Iterable, List

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion


from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)




# Model saving
import os
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True)
random.seed(42)
np.random.seed(42)

print("Libraries Imported!")

Libraries Imported!


In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "urbanpulse_311_complaints.csv"
#DATA_PATH = PROJECT_ROOT / "data" / "raw" / "urbanpulse_311_complaints_bilingual.csv"
OUTPUT_DIR = PROJECT_ROOT / "models" / "model4_nlp_classification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)

Project root: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting
Data path: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\data\raw\urbanpulse_311_complaints.csv
Output dir: C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\models\model4_nlp_classification


In [3]:
df = pd.read_csv(DATA_PATH)
print("Data shape:", df.shape)
#df.head(3)

Data shape: (434722, 11)


In [4]:
# Lowercase text columns only
df = df.apply(lambda col: col.str.lower() if col.dtype == "object" else col)

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-/&]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:


def get_top_complaint_types(df: pd.DataFrame, n: int = 5) -> list:
    return df["complaint_type"].value_counts().head(n).index.tolist()

TOP_5 = get_top_complaint_types(df, n=5)
TOP_5

['illegal parking',
 'heat/hot water',
 'noise - residential',
 'snow or ice',
 'blocked driveway']

In [6]:
def create_complaint_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map complaint types to the top 5 categories + "Other" (6 classes total).

    The top 5 categories are:
    - Illegal Parking
    - HEAT/HOT WATER
    - Noise - Residential
    - Snow or Ice
    - Blocked Driveway

    Everything else maps to "Other". This gives you 6 classes total —
    a much more manageable classification problem than 186 categories.
    """
    top_5 = ['illegal parking', 'heat/hot water', 'noise - residential',
             'snow or ice', 'blocked driveway']
    df['complaint_category'] = df['complaint_type'].apply(
        lambda x: x if x in top_5 else 'other'
    )

    print("Complaint category distribution:")
    print(df['complaint_category'].value_counts())

    coverage = df[df['complaint_category'] != 'other'].shape[0] / len(df) * 100
    print(f"\nTop 5 categories cover {coverage:.1f}% of all complaints")
    print(f"Total classes: {df['complaint_category'].nunique()} (top 5 + other)")

    return df

In [7]:



# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------
SEED = 42
TEST_SIZE = 0.20

DEFAULT_ROUTE_MAP = {
    "blocked driveway": "nypd",
    "heat/hot water": "hpd",
    "illegal parking": "nypd",
    "noise - residential": "nypd",
    "snow or ice": "dsny",
}

#TEXT_COLS = ["complaint_type", "descriptor", "resolution_description"]
TEXT_COLS = ["descriptor", "resolution_description"]

In [8]:

# ---------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------
def setup_logging() -> logging.Logger:
    logger = logging.getLogger("model4_train")
    logger.setLevel(logging.INFO)

    if logger.handlers:
        return logger

    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.propagate = False
    return logger


LOGGER = setup_logging()

In [9]:


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def safe_str(x) -> str:
    return "" if pd.isna(x) else str(x).strip()



In [10]:


LOGGER.info("Starting preprocessing")

df = df.copy()

for col in TEXT_COLS:
    if col not in df.columns:
        LOGGER.warning("Missing column '%s'; creating empty fallback", col)
        df[col] = ""
    df[col] = df[col].fillna("")

df = create_complaint_categories(df)

LOGGER.info("Building complaint_text field")
df["complaint_text"] = (df["descriptor"].map(clean_text) + " | " + df["resolution_description"].map(clean_text))

#df["complaint_text"] = (df["resolution_description"].map(clean_text))
df["categories"] = df["complaint_category"].fillna("").map(clean_text)

before = len(df)
df = df[(df["complaint_text"] != "") & (df["categories"] != "")].copy()
df = df.reset_index(drop=True)
after = len(df)

LOGGER.info("Filtered empty rows: %s -> %s", before, after)
LOGGER.info("Finished preprocessing")

2026-04-18 21:53:39 | INFO | Starting preprocessing
Complaint category distribution:
complaint_category
other                  212897
illegal parking         66159
heat/hot water          64362
noise - residential     38931
snow or ice             27453
blocked driveway        24920
Name: count, dtype: int64

Top 5 categories cover 51.0% of all complaints
Total classes: 6 (top 5 + other)
2026-04-18 21:53:39 | INFO | Building complaint_text field
2026-04-18 21:54:01 | INFO | Filtered empty rows: 434722 -> 434722
2026-04-18 21:54:01 | INFO | Finished preprocessing


In [11]:


# =========================================================
# 3. PREP DATA
# =========================================================
# Example assumptions:
# df has:
#   - 'resolution_description'
#   - 'categories'   (target labels)

df_model = df.copy()

df_model["complaint_text"] = df_model["complaint_text"].fillna("").map(clean_text)
df_model = df_model[(df_model["complaint_text"] != "") & (df_model["categories"].notna())].copy()

print("Modeling shape:", df_model.shape)
print(df_model["categories"].value_counts())

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_model["categories"])

X_train_text, X_val_text, y_train, y_val = train_test_split(
    df_model["complaint_text"],
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Modeling shape: (434722, 14)
categories
other                  212897
illegal parking         66159
heat/hot water          64362
noise - residential     38931
snow or ice             27453
blocked driveway        24920
Name: count, dtype: int64


In [12]:
#!pip install nltk

#import nltk
#nltk.download("stopwords")

#from nltk.corpus import stopwords
#from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [13]:



# =========================================================
# 1. Custom transformer for extra keyword features
# =========================================================
class ExtraTextFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        s = pd.Series(X).fillna("").astype(str).str.lower()

        extra = pd.DataFrame({
            "is_driveway": s.str.contains(r"\bdriveway\b", regex=True).astype(int),
            "is_parking": s.str.contains(r"\bparking\b|\bparked\b|\bvehicle\b|\bcar\b", regex=True).astype(int),
            "is_blocked": s.str.contains(r"\bblocked\b|\bblocking\b", regex=True).astype(int),
            "is_noise": s.str.contains(r"\bbanging\b|\bpounding\b|\bloud\b|\bmusic\b", regex=True).astype(int),
        })

        return csr_matrix(extra.values)


# =========================================================
# 2. Full pipeline
# =========================================================
model_pipeline = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word_tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 3),
                    max_features=12000,
                    min_df=10,
                    max_df=0.95,
                    # stop_words=STOP_WORDS,
                    sublinear_tf=True
                )
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 4),
                    max_features=5000,
                    min_df=10,
                    max_df=0.95,
                    sublinear_tf=True
                )
            ),
            (
                "extra_features",
                ExtraTextFeatures()
            ),
        ])
    ),
    (
        "clf",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-6,
            max_iter=1000,
            tol=1e-4,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5,
            random_state=42,
            n_jobs=-1
        )
    )
])


# =========================================================
# 3. Fit / predict
# =========================================================
model_pipeline.fit(X_train_text, y_train)

y_pred = model_pipeline.predict(X_val_text)

# optional probabilities / confidence
y_proba = model_pipeline.predict_proba(X_val_text)
y_conf = y_proba.max(axis=1)

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

acc = accuracy_score(y_val, y_pred)
f1_weighted = f1_score(y_val, y_pred, average="weighted")

print("Accuracy:", round(acc, 4))
print("Weighted F1:", round(f1_weighted, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

cm = confusion_matrix(y_val, y_pred)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print("\nConfusion Matrix:")
print(cm_df)

Accuracy: 0.969
Weighted F1: 0.9702

Classification Report:
                     precision    recall  f1-score   support

   blocked driveway       1.00      1.00      1.00      4984
     heat/hot water       1.00      1.00      1.00     12872
    illegal parking       1.00      1.00      1.00     13232
noise - residential       0.77      0.94      0.84      7786
              other       0.99      0.95      0.97     42580
        snow or ice       1.00      1.00      1.00      5491

           accuracy                           0.97     86945
          macro avg       0.96      0.98      0.97     86945
       weighted avg       0.97      0.97      0.97     86945


Confusion Matrix:
                     blocked driveway  heat/hot water  illegal parking  \
blocked driveway                 4984               0                0   
heat/hot water                      0           12872                0   
illegal parking                     0               0            13232   
noise - resi

In [16]:
LOGGER.info("Evaluating %s", "Model")

print(f"\n=== Model Evaluation ===")
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(
        f"Precision (weighted): "
        f"{precision_score(y_val, y_pred, average='weighted', zero_division=0):.4f}"
    )
print(
        f"Recall (weighted): "
        f"{recall_score(y_val, y_pred, average='weighted', zero_division=0):.4f}"
    )
print(
        f"F1 (weighted): "
        f"{f1_score(y_val, y_pred, average='weighted', zero_division=0):.4f}"
    )

2026-04-18 22:05:03 | INFO | Evaluating Model

=== Model Evaluation ===
Accuracy: 0.9690
Precision (weighted): 0.9734
Recall (weighted): 0.9690
F1 (weighted): 0.9702


# Build Complaint Routing Recommendations

Use the same Training TEXT to predict which "agency" should the complaint be routed to. 
Such as:

**"blocked driveway"** --- nypd

**"heat/hot water"** --- hpd

**"illegal parking"** --- nypd

**"noise - residential"** --- nypd

**"snow or ice"** --- dsny

In [17]:
# -----------------------------
# Encode labels for route-to-agency
# -----------------------------
df_model2 = df_model.copy()

label_encoder_rt = LabelEncoder()
df_model2["rt_label"] = label_encoder_rt.fit_transform(df_model2["agency"])

id2label_rt = {i: label for i, label in enumerate(label_encoder_rt.classes_)}
label2id_rt = {label: i for i, label in id2label_rt.items()}

print("\nLabel mapping:")
print(id2label_rt)


Label mapping:
{0: 'dcwp', 1: 'dep', 2: 'dhs', 3: 'dob', 4: 'doe', 5: 'dohmh', 6: 'dot', 7: 'dpr', 8: 'dsny', 9: 'hpd', 10: 'nypd', 11: 'oos', 12: 'oti', 13: 'tlc'}


In [18]:
## use complaint_type to predict route_to_agency for "other" category

X_train_text_rt, X_val_text_rt, y_train_rt, y_test_rt = train_test_split(
    df_model2["complaint_text"],
    df_model2["rt_label"],
    test_size=0.2,
    random_state=42,
    stratify=df_model2["rt_label"]
)

In [19]:
# =========================================================
# 2. Full pipeline
# =========================================================
model_pipeline_rt = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word_tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 3),
                    max_features=12000,
                    min_df=10,
                    max_df=0.95,
                    # stop_words=STOP_WORDS,
                    sublinear_tf=True
                )
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 4),
                    max_features=5000,
                    min_df=10,
                    max_df=0.95,
                    sublinear_tf=True
                )
            ),
            (
                "extra_features",
                ExtraTextFeatures()
            ),
        ])
    ),
    (
        "clf",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=20,
            tol=1e-3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    )
])


# =========================================================
# 3. Fit / predict
# =========================================================
model_pipeline_rt.fit(X_train_text_rt, y_train_rt)

y_pred_rt = model_pipeline_rt.predict(X_val_text_rt)

# optional probabilities / confidence
y_proba_rt = model_pipeline_rt.predict_proba(X_val_text_rt)
y_conf_rt = y_proba_rt.max(axis=1)

In [21]:


acc_rt = accuracy_score(y_test_rt, y_pred_rt)
f1_weighted = f1_score(y_test_rt, y_pred_rt, average="weighted")

print("Accuracy:", round(acc, 4))
print("Weighted F1:", round(f1_weighted, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test_rt,
        y_pred_rt,
        target_names=label_encoder_rt.classes_,
        zero_division=0
    )
)


Accuracy: 0.969
Weighted F1: 0.9998

Classification Report:
              precision    recall  f1-score   support

        dcwp       1.00      1.00      1.00       383
         dep       1.00      1.00      1.00      4144
         dhs       1.00      1.00      1.00       456
         dob       1.00      1.00      1.00      2225
         doe       1.00      1.00      1.00        66
       dohmh       1.00      1.00      1.00      1291
         dot       1.00      1.00      1.00      7728
         dpr       1.00      1.00      1.00      1436
        dsny       1.00      1.00      1.00     10296
         hpd       1.00      1.00      1.00     25570
        nypd       1.00      1.00      1.00     32700
         oos       0.10      1.00      0.18         1
         oti       0.60      1.00      0.75         3
         tlc       1.00      1.00      1.00       646

    accuracy                           1.00     86945
   macro avg       0.91      1.00      0.92     86945
weighted avg       1

In [ ]:
## Save models and artifacts

model_path = "../models/model4_nlp_classification/saved_model/"


joblib.dump(model_pipeline, model_path + "model4_complaint_classifier_char_tfidf_SGD.pkl")
joblib.dump(model_pipeline_rt, model_path + "model4_routing_classifier_char_tfidf_SGD.pkl")
joblib.dump(label_encoder, model_path + "model4_category_label_encoder.pkl")
joblib.dump(label_encoder_rt, model_path + "model4_routing_label_encoder.pkl")

print("Saved artifacts to:", model_path)

Saved artifacts to: ../models/model4_nlp_classification/


In [23]:
test1 = {"id":1,
         "descriptor": "Double Parked Blocking Traffic",
         "resolution_description": "The New York City Police Department responded to the complaint and with the information available observed no evidence of a criminal violation at that time. If the problem persists, please contact 311 to create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. If necessary, your complaint may be referred to your local precinct's special operations units (Quality of Life, etc.). We count on New Yorkers like yourself to maintain a safe City, so please let us know if you see other conditions that require our attention."

}

In [24]:
test_df = pd.DataFrame([test1]).set_index("id")

In [27]:
pred = model_pipeline.predict(test1)
pred_label = label_encoder.inverse_transform(pred)

In [28]:
print(pred_label)

['other' 'illegal parking' 'other']


In [30]:
from scipy.sparse import hstack, csr_matrix

test1 = {
    "id": 1,
    "descriptor": "Double Parked Blocking Traffic",
    "resolution_description": "The New York City Police Department responded to the complaint and with the information available observed no evidence of the violation."
}

test_df = pd.DataFrame([test1])

# 1. Build complaint_text the same way as training
test_df["complaint_text"] = (
    test_df["descriptor"].fillna("").astype(str).str.lower() + " " +
    test_df["resolution_description"].fillna("").astype(str).str.lower()
)

# 2. Transform with saved vectorizer
X_test_tfidf = vectorizer.transform(test_df["complaint_text"])

# 3. If your model used extra features, add them too
test_extra = pd.DataFrame({
    "is_driveway": test_df["complaint_text"].str.contains(r"\bdriveway\b", regex=True).astype(int),
    "is_parking": test_df["complaint_text"].str.contains(r"\bparking\b|\bparked\b|\bvehicle\b|\bcar\b", regex=True).astype(int),
    "is_blocked": test_df["complaint_text"].str.contains(r"\bblocked\b|\bblocking\b", regex=True).astype(int),
    "is_noise": test_df["complaint_text"].str.contains(r"\bbanging\b|\bpounding\b|\bloud\b|\bmusic\b", regex=True).astype(int),
})

X_test = hstack([X_test_tfidf, csr_matrix(test_extra.values)])

# 4. Predict
pred = model.predict(X_test)
pred_label = label_encoder.inverse_transform(pred)

print(pred_label)

['illegal parking']


In [31]:
test2 = {
    "id": 2,
    "descriptor": "Banging/Pounding",
    "resolution_description": "The New York City Police Department responded to the complaint and their investigation determined that \
    no criminal violation existed. \
    The condition was corrected without the need to issue a summons or effect an arrest. If the problem persists, please contact 311 to \
    create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. \
    If necessary, your complaint may be referred to your local precinct's special operations units (Quality of Life, etc.). \
    Thank you for your attention to this matter. We count on New Yorkers like yourself to maintain a safe City, so please let us know \
    if you see other conditions that require our attention."
}

test_df = pd.DataFrame([test2])

# 1. Build complaint_text the same way as training
test_df["complaint_text"] = (
    test_df["descriptor"].fillna("").astype(str).str.lower() + " " +
    test_df["resolution_description"].fillna("").astype(str).str.lower()
)

# 2. Transform with saved vectorizer
X_test_tfidf = vectorizer.transform(test_df["complaint_text"])

# 3. If your model used extra features, add them too
test_extra = pd.DataFrame({
    "is_driveway": test_df["complaint_text"].str.contains(r"\bdriveway\b", regex=True).astype(int),
    "is_parking": test_df["complaint_text"].str.contains(r"\bparking\b|\bparked\b|\bvehicle\b|\bcar\b", regex=True).astype(int),
    "is_blocked": test_df["complaint_text"].str.contains(r"\bblocked\b|\bblocking\b", regex=True).astype(int),
    "is_noise": test_df["complaint_text"].str.contains(r"\bbanging\b|\bpounding\b|\bloud\b|\bmusic\b", regex=True).astype(int),
})

X_test = hstack([X_test_tfidf, csr_matrix(test_extra.values)])

# 4. Predict
pred = model.predict(X_test)
pred_label = label_encoder.inverse_transform(pred)

print(pred_label)

['other']


In [34]:
test3 = {
    "id": 3,
    "descriptor": "Calle no barrida",
    "resolution_description": "El Departamento de Saneamiento investigÃ³ esta denuncia y no encontrÃ³ ninguna condiciÃ³n en el lugar."
}

test_df = pd.DataFrame([test3])

# 1. Build complaint_text the same way as training
test_df["complaint_text"] = (
    test_df["descriptor"].fillna("").astype(str).str.lower() + " " +
    test_df["resolution_description"].fillna("").astype(str).str.lower()
)

# 2. Transform with saved vectorizer
X_test_tfidf = vectorizer.transform(test_df["complaint_text"])

# 3. If your model used extra features, add them too
test_extra = pd.DataFrame({
    "is_driveway": test_df["complaint_text"].str.contains(r"\bdriveway\b", regex=True).astype(int),
    "is_parking": test_df["complaint_text"].str.contains(r"\bparking\b|\bparked\b|\bvehicle\b|\bcar\b", regex=True).astype(int),
    "is_blocked": test_df["complaint_text"].str.contains(r"\bblocked\b|\bblocking\b", regex=True).astype(int),
    "is_noise": test_df["complaint_text"].str.contains(r"\bbanging\b|\bpounding\b|\bloud\b|\bmusic\b", regex=True).astype(int),
})

X_test = hstack([X_test_tfidf, csr_matrix(test_extra.values)])

# 4. Predict
pred = model_rt.predict(X_test)
pred_label = label_encoder_rt.inverse_transform(pred)

print(pred_label)

['oos']
